# Pandas Data Science Lab — School Store Transactions
**Name:** _______________________  **Date:** _______________

Work through **26 core questions** and **8 optional advanced challenges**. The dataset is synthetic, created in this notebook; no separate CSV download is required.

**Rules:** Run the setup cell. Answer each question in its blank code cell, keep outputs visible, and explain results when asked. Use the slide deck as a reference. Do not modify the raw `orders` object unless a question explicitly asks; make copies for cleaning.

## Dataset setup (run this cell once)
Each row represents a transaction. Columns include order date, region, product, category, unit price, units, discount, payment method, member status, age, rating, and shipping days. The dataset intentionally includes a few duplicates, missing values, inconsistent region spellings, and invalid unit counts.

In [ ]:
import pandas as pd
import numpy as np

# Reproducible fictional school-store transactions.
rng = np.random.default_rng(2026)
products = ["Notebook", "Calculator", "Backpack", "Water Bottle",
            "Planner", "Markers", "Binder", "Laptop Sleeve"]
categories = ["Paper", "Electronics", "Bags", "Accessories",
              "Paper", "Art", "Paper", "Accessories"]
prices = [4.50, 18.00, 32.00, 12.00, 8.00, 6.50, 5.00, 22.00]
costs = [1.80, 10.50, 19.00, 5.00, 3.40, 2.60, 1.70, 12.00]

n = 180
product_ids = rng.integers(0, len(products), size=n)
orders = pd.DataFrame({
    "Order_ID": [f"S{1000+i}" for i in range(n)],
    "Order_Date": pd.to_datetime(rng.choice(
        pd.date_range("2026-09-01", periods=120), size=n
    )).strftime("%Y-%m-%d"),
    "Product": np.array(products)[product_ids],
    "Category": np.array(categories)[product_ids],
    "Unit_Price": np.array(prices)[product_ids],
    "Units": rng.integers(1, 7, size=n),
    "Discount": rng.choice([0.00, 0.05, 0.10, 0.20],
                           size=n, p=[0.48, 0.20, 0.24, 0.08]),
    "Region": rng.choice(["North", "South", "East", "West",
                          "north", " EAST ", "west "], size=n),
    "Payment_Method": rng.choice(["Card", "Cash", "Online"], size=n),
    "Member": rng.choice(["Yes", "No"], size=n),
    "Customer_Age": rng.integers(15, 66, size=n).astype(float),
    "Rating": rng.integers(1, 6, size=n).astype(float),
    "Shipping_Days": rng.integers(1, 9, size=n)
})

# Intentional data-quality issues for cleaning practice.
orders.loc[[5, 22, 79, 114, 156], "Rating"] = np.nan
orders.loc[[12, 41, 92], "Customer_Age"] = np.nan
orders.loc[[17, 83], "Units"] = -2
orders = pd.concat([orders, orders.iloc[[4, 25, 90, 140]]],
                   ignore_index=True)

# Two reference tables for the advanced merge problems.
product_lookup = pd.DataFrame({
    "Product": products,
    "Supplier_Cost": costs
})
region_targets = pd.DataFrame({
    "Region": ["North", "South", "East", "West"],
    "Revenue_Target": [1700, 1700, 1700, 1700]
})

print("Dataset ready!", orders.shape)
print(orders.head(3))

## Only completed example — selecting a column
A Pandas column is selected using brackets and its name. This displays the first five products; now complete the rest yourself.

In [ ]:
orders["Product"].head()

# Part A — Inspecting and Selecting Data

### 1. Inspect the dataset
Display the first 8 rows and the last 5 rows. How many rows and columns are present?

*Syntax reference:* `head()`, `tail()`, `shape`

### 2. Describe columns
List all column names, inspect data types, and run a compact dataset summary.

*Syntax reference:* `columns`, `dtypes`, `info()`

### 3. Select columns
Show only `Order_ID`, `Product`, `Units`, and `Unit_Price` for the first 10 rows.

*Syntax reference:* `df[["A", "B"]]`, `head()`

### 4. Select with loc and iloc
Use `.loc` to retrieve the product at row label 7 and `.iloc` to retrieve the value at row position 7 in column position 2. Explain why the two answers agree.

*Syntax reference:* `.loc[row_label, "Column"]`, `.iloc[row_pos, col_pos]`

### 5. Find the unique categories
Display all distinct product categories and calculate how many transactions belong to each.

*Syntax reference:* `unique()`, `value_counts()`

### 6. Order size distribution
Count how many orders have each `Units` value. Do any values seem invalid?

*Syntax reference:* `value_counts()`, `sort_index()`

### 7. Sort by price
Display the 12 transactions with the highest unit prices, including their products and regions.

*Syntax reference:* `sort_values(..., ascending=False)`, `head()`

### 8. Save and reload a CSV
Save the current dataset to a CSV without its index, reload it, and verify the shape matches.

*Syntax reference:* `to_csv(..., index=False)`, `pd.read_csv()`

# Part B — Filtering, New Columns, and Summaries

### 9. Large purchases
Filter orders containing at least 5 units. How many rows remain?

*Syntax reference:* Boolean mask and `len()`

### 10. Two conditions
Find transactions from the `North` region where the unit price exceeds $10. Explain why lowercase `north` is not included yet.

*Syntax reference:* `&`, parentheses around each condition

### 11. Either condition
Find transactions where payment was `Cash` OR the customer was a member.

*Syntax reference:* `|`, parentheses

### 12. Gross revenue
Create `Gross_Revenue` = `Units × Unit_Price`. Show the first five results.

*Syntax reference:* Column assignment and vectorized `*`

### 13. Revenue after discounts
Create `Net_Revenue` = `Units × Unit_Price × (1 − Discount)`.

*Syntax reference:* Vectorized arithmetic

### 14. Descriptive statistics
Summarize `Unit_Price`, `Units`, and `Net_Revenue` with numerical descriptive statistics.

*Syntax reference:* `describe()`

### 15. Totals and averages
Calculate the total net revenue and average number of units per transaction. Are the negative-unit rows a concern?

*Syntax reference:* `sum()`, `mean()`

**Interpretation:** *Write your explanation here.*

### 16. Group by product category
Calculate average unit price and total net revenue for each product category.

*Syntax reference:* `groupby()`, `mean()`, `sum()`

### 17. Best-selling products
Find the top three products by total units sold. Use a grouped sum and sort.

*Syntax reference:* `groupby()`, `sum()`, `sort_values()`

# Part C — Cleaning and Interpreting Data

### 18. Check missing values
Count missing values in every column. Identify which variables need attention.

*Syntax reference:* `isna().sum()`

### 19. Identify duplicate rows
Count exact duplicate records and display the duplicated rows.

*Syntax reference:* `duplicated(keep=False)`, `sum()`

### 20. Remove duplicates safely
Create a NEW DataFrame `clean_orders` with exact duplicates removed. Verify row counts before and after.

*Syntax reference:* `drop_duplicates()`, `.copy()`

### 21. Handle missing ratings
Create a new `Rating_Filled` column in `clean_orders` using the median rating. Keep the original Rating column for comparison.

*Syntax reference:* `median()`, `fillna()`

### 22. Flag bad unit counts
Find nonpositive `Units` values, then create `valid_orders` containing only positive-unit records.

*Syntax reference:* `df[df["Units"] > 0]`

### 23. Recalculate valid revenue
Create `Net_Revenue` for `valid_orders` and calculate the total. Compare it with the uncleaned total from Question 15.

*Syntax reference:* Column assignment, `sum()`

**Interpretation:** *Write your explanation here.*

### 24. Compare regions
Calculate total valid net revenue by `Region`. Are some regions repeated because of inconsistent spelling?

*Syntax reference:* `groupby()`, `sum()`

### 25. Customer satisfaction
Calculate average rating for each payment method, ignoring missing ratings. Which payment method has the highest mean in these fictional data?

*Syntax reference:* `groupby()["Rating"].mean()`

### 26. Write a short finding
Write 3–4 sentences interpreting TWO meaningful results. Mention at least one data-quality issue that could affect conclusions.

*Syntax reference:* Use a Markdown cell below your code

**Interpretation:** *Write your explanation here.*

# Part D — Optional Advanced Challenges

### 27. Normalize category strings
Create a cleaned Region column so `north`, `North`, ` EAST `, and similar values become consistent. Compare counts before and after.

*Syntax reference:* New syntax: `.str.strip().str.title()`

### 28. Analyze monthly revenue
Convert Order_Date into a datetime column, then group net revenue by month and identify the strongest month.

*Syntax reference:* New syntax: `pd.to_datetime()`, `.dt.to_period("M")`

**Interpretation:** *Write your explanation here.*

### 29. Multiple group statistics
For each cleaned region, calculate total revenue, number of orders, and average shipping days in ONE table.

*Syntax reference:* New syntax: `.groupby(...).agg(total=("Net_Revenue", "sum"), ...)`

### 30. Build a pivot table
Create a table of summed net revenue, with cleaned regions as rows and Category as columns; fill missing combinations with 0.

*Syntax reference:* New syntax: `pd.pivot_table(data, values=..., index=..., columns=..., aggfunc="sum", fill_value=0)`

### 31. Merge supplier costs
Merge valid orders with `product_lookup`, calculate per-order estimated profit after discounts, and identify the category with highest total profit.

*Syntax reference:* New syntax: `.merge(product_lookup, on="Product", how="left")`

**Interpretation:** *Write your explanation here.*

### 32. Numeric correlation
Calculate correlations among Units, Unit_Price, Rating, Shipping_Days, and Net_Revenue. Explain why correlation does not establish causation.

*Syntax reference:* New syntax: `df[[...]].corr()`

**Interpretation:** *Write your explanation here.*

### 33. Daily revenue and rolling average
Create one revenue total per date, sort chronologically, and calculate a rolling average over 7 recorded dates. Make a line plot if you have time.

*Syntax reference:* New syntax: `.groupby("Order_Date").sum()`, `.rolling(7).mean()`

### 34. Customer behavior cross-tab
Build a frequency table of Member status against Payment_Method. Then calculate percentages by member group.

*Syntax reference:* New syntax: `pd.crosstab(..., normalize="index")`

## End-of-lab reflection
1. Which Pandas operations were most useful?
2. How did cleaning change the analysis?
3. Which advanced challenge would you use in a data science project?